### 7 · Reranking — Improving Precision After Retrieval

Vector similarity search is fast but imprecise — it uses **bi-encoder** embeddings
that compress meaning into a single vector. Reranking adds a second pass with a
**cross-encoder** that scores each (query, document) pair jointly, dramatically
improving precision.

### What you'll learn
```
1. Why reranking matters (bi-encoder vs cross-encoder tradeoff)
2. LLM-based reranking (no extra model needed)
3. Building a rerank node into the LangGraph pipeline
4. Comparing retrieval quality: with vs without reranking
```

### The Pattern
```
Retrieve top-20 (high recall)  →  Rerank  →  Keep top-5 (high precision)
```
Over-retrieve then prune. Best of both worlds.

In [ ]:
from pathlib import Path
from typing import TypedDict

from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langgraph.graph import END, StateGraph
from dotenv import load_dotenv

load_dotenv(override=True)

VECTORSTORE_DIR = Path("../data/vectorstore")
EMBEDDING_MODEL = "text-embedding-3-small"
LLM_MODEL = "gpt-4o-mini"

embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)
vectorstore = FAISS.load_local(
    str(VECTORSTORE_DIR), embeddings, allow_dangerous_deserialization=True
)
llm = ChatOpenAI(model=LLM_MODEL, temperature=0)

print(f"Loaded {vectorstore.index.ntotal} vectors.")

---
### Step 1 — The Problem: Why Reranking?

Let's see how similarity search can return irrelevant results in top-k.

In [ ]:
query = "What are the anti-patterns to avoid in Redux state management?"

# Retrieve top 15 — some will be relevant, some noisy
results = vectorstore.similarity_search_with_relevance_scores(query, k=15)

print(f"Query: {query}\n")
print(f"{'Rank':<6} {'Score':<8} {'Source':<45} Section")
print("-" * 110)
for i, (doc, score) in enumerate(results):
    print(f"{i+1:<6} {score:.4f}  {doc.metadata.get('source','?'):<45} {doc.metadata.get('section','?')[:40]}")

print(f"\nNotice: lower-ranked results may be from unrelated docs.")
print("Reranking will rescore these and push the best ones to the top.")

---
### Step 2 — LLM-Based Reranker

We use the LLM itself as a cross-encoder reranker. For each retrieved chunk,
the LLM scores how relevant it is to the query on a 0-10 scale.

**Why LLM reranking?**
- No extra model to install or manage
- Works great with `gpt-4o-mini` (cheap and fast)
- In production, you'd use Cohere Rerank or a local cross-encoder for cost efficiency

In [ ]:
import json as json_mod

RERANK_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "You are a relevance scoring system. Given a query and a document chunk, "
     "score how relevant the chunk is to answering the query.\n\n"
     "Score from 0 to 10:\n"
     "- 0: Completely irrelevant\n"
     "- 5: Somewhat relevant but not directly answering\n"
     "- 10: Directly answers the query\n\n"
     "Respond with ONLY a JSON object: {{\"score\": <int>, \"reason\": \"brief explanation\"}}"),
    ("human",
     "Query: {query}\n\nDocument chunk:\n{chunk}"),
])


def rerank_documents(query: str, docs: list[Document], top_n: int = 5) -> list[tuple[Document, int, str]]:
    """Rerank documents using LLM scoring. Returns (doc, score, reason) sorted by score."""
    chain = RERANK_PROMPT | llm | StrOutputParser()
    scored = []

    for doc in docs:
        try:
            resp = chain.invoke({"query": query, "chunk": doc.page_content[:500]})
            cleaned = resp.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
            parsed = json_mod.loads(cleaned)
            scored.append((doc, parsed["score"], parsed.get("reason", "")))
        except Exception as e:
            scored.append((doc, 0, f"Parse error: {e}"))

    scored.sort(key=lambda x: x[1], reverse=True)
    return scored[:top_n]


print("LLM reranker ready.")

In [ ]:
# ── Demo: Rerank the top-15 results ──────────────────────────────────────────

query = "What are the anti-patterns to avoid in Redux state management?"
raw_docs = vectorstore.similarity_search(query, k=15)

print(f"Query: {query}")
print(f"Retrieved {len(raw_docs)} chunks, reranking...\n")

reranked = rerank_documents(query, raw_docs, top_n=5)

print(f"{'Rank':<6} {'Score':<7} {'Source':<45} Reason")
print("-" * 110)
for i, (doc, score, reason) in enumerate(reranked):
    print(f"{i+1:<6} {score:<7} {doc.metadata.get('source','?'):<45} {reason[:50]}")

print(f"\nReranking kept the most relevant 5 out of 15 candidates.")

---
### Step 3 — Build Graph with Reranking Node

```
START → retrieve_docs (k=15) → rerank_docs (keep top-5) → generate_answer → END
```

In [ ]:
class GraphState(TypedDict):
    question: str
    documents: list[Document]         # after retrieval (k=15)
    reranked_documents: list[Document] # after reranking (top-5)
    answer: str


# ── Node: Retrieve (over-retrieve) ────────────────────────────────────────────
wide_retriever = vectorstore.as_retriever(search_kwargs={"k": 15})

def retrieve_docs(state: GraphState) -> dict:
    docs = wide_retriever.invoke(state["question"])
    print(f"  [retrieve] → {len(docs)} candidates")
    return {"documents": docs}


# ── Node: Rerank ──────────────────────────────────────────────────────────────

def rerank_docs(state: GraphState) -> dict:
    reranked = rerank_documents(state["question"], state["documents"], top_n=5)
    top_docs = [doc for doc, score, reason in reranked]
    print(f"  [rerank] → kept top {len(top_docs)} (scores: {[s for _, s, _ in reranked]})")
    return {"reranked_documents": top_docs}


# ── Node: Generate ────────────────────────────────────────────────────────────

ANSWER_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "You are a helpful assistant for the AMS Admin Tool team. "
     "Use the context to answer accurately. If info is missing, say so.\n\n"
     "Context:\n{context}"),
    ("human", "{question}"),
])

def generate_answer(state: GraphState) -> dict:
    # Use reranked docs (not raw retrieval)
    context = "\n\n---\n\n".join(
        f"[{d.metadata.get('source', '?')}]\n{d.page_content}"
        for d in state["reranked_documents"]
    )
    chain = ANSWER_PROMPT | llm | StrOutputParser()
    answer = chain.invoke({"question": state["question"], "context": context})
    print(f"  [generate] → {len(answer)} chars")
    return {"answer": answer}


# ── Wire Graph ─────────────────────────────────────────────────────────────────

workflow = StateGraph(GraphState)
workflow.add_node("retrieve_docs", retrieve_docs)
workflow.add_node("rerank_docs", rerank_docs)
workflow.add_node("generate_answer", generate_answer)

workflow.set_entry_point("retrieve_docs")
workflow.add_edge("retrieve_docs", "rerank_docs")
workflow.add_edge("rerank_docs", "generate_answer")
workflow.add_edge("generate_answer", END)

rerank_graph = workflow.compile()
print("Reranking graph compiled.")

In [ ]:
from IPython.display import Image, display
display(Image(rerank_graph.get_graph().draw_mermaid_png()))

---
### Step 4 — Compare: With vs Without Reranking

Side-by-side comparison on the same query.

In [ ]:
# ── Baseline (no reranking, k=5) ──────────────────────────────────────────────

baseline_retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
baseline_chain = ANSWER_PROMPT | llm | StrOutputParser()

TEST_QUERIES = [
    "What are the anti-patterns to avoid in Redux state management?",
    "How does the theming system work with Material UI?",
    "What Firestore security rules should be implemented?",
]

for query in TEST_QUERIES:
    print(f"\nQuery: {query}")
    print("=" * 80)

    # Baseline
    baseline_docs = baseline_retriever.invoke(query)
    baseline_ctx = "\n\n---\n\n".join(d.page_content for d in baseline_docs)
    baseline_answer = baseline_chain.invoke({"question": query, "context": baseline_ctx})

    # Reranking
    rerank_result = rerank_graph.invoke({"question": query})

    print(f"\n--- BASELINE (k=5) sources ---")
    for d in baseline_docs:
        print(f"  {d.metadata.get('source','?')}")

    print(f"\n--- RERANKED (15→5) sources ---")
    for d in rerank_result["reranked_documents"]:
        print(f"  {d.metadata.get('source','?')}")

    print(f"\nBaseline answer: {baseline_answer[:200]}...")
    print(f"Reranked answer: {rerank_result['answer'][:200]}...")

---
### Takeaways

| Concept | What you learned |
|---------|------------------|
| **Bi-encoder vs Cross-encoder** | Embeddings are fast/approximate; rerankers are slow/precise |
| **Over-retrieve + Prune** | Retrieve k=15-20, rerank, keep top-5 |
| **LLM-as-reranker** | Works well with gpt-4o-mini, no extra model |
| **Rerank node** | Clean separation in LangGraph — easy to swap implementations |

**Production upgrades:**
- Use **Cohere Rerank API** or a local **cross-encoder** model for lower cost
- Batch rerank requests to reduce latency
- Combine with metadata filtering: filter first → retrieve → rerank